# Layer 06 - RAG Property Search (Ollama Gemma3)

This notebook reproduces the `03_rag_search` workflow using **Ollama Gemma3** via `PropertyRAGSearchGemma` from `yc_rag_search_gemma.py`.

## Senior AI Engineer Workflow

1. **Environment and dependency readiness**
   - Verify Neo4j connectivity and `.env` credentials.
   - Ensure Ollama daemon is running and Gemma model is pulled.
2. **Stage 1 quality gate (NL -> Params)**
   - Validate extraction reliability before full execution.
   - Check fallback behavior and parsed JSON consistency.
3. **Stage 2 retrieval validation (Params -> Search)**
   - Run weighted and graph-traversal scenarios.
   - Confirm filters and ranking align with user intent.
4. **Stage 3 grounded generation (Results -> Answer)**
   - Ensure answers reference actual returned properties.
   - Validate graceful degradation for empty/noisy outputs.
5. **Operational hardening**
   - Test edge cases, impossible filters, and ambiguous prompts.
   - Keep cleanup deterministic with explicit connection close.

## Prerequisites

- `01_build_knowledge_base.ipynb` already executed.
- `.env` at repository root with:
  - `NEO4J_URI`
  - `NEO4J_USERNAME`
  - `NEO4J_PASSWORD`
  - `NEO4J_DATABASE`
- Ollama running locally:
  - `ollama serve`
  - `ollama pull gemma:7b-instruct`

## Suggested packages

```bash
pip install pandas python-dotenv neo4j requests langchain-community
```

In [1]:
# --- Path setup (standard PropertyLens pattern) ---
import sys
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / 'hf_data').exists() else cwd.parent
LAYER_DIR = REPO_ROOT / '06_search_layer'

if str(LAYER_DIR) not in sys.path:
    sys.path.insert(0, str(LAYER_DIR))

load_dotenv(REPO_ROOT / '.env')

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'LAYER_DIR : {LAYER_DIR}')

REPO_ROOT : d:\Master Degree\Projects\Transparent_AI\PropertyLens
LAYER_DIR : d:\Master Degree\Projects\Transparent_AI\PropertyLens\06_search_layer


## 1. Initialize Gemma3-backed RAG search

We use `PropertyRAGSearchGemma` directly, which internally connects to Ollama and Neo4j.

If your model tag differs, update `model_name` accordingly.

In [2]:
import requests

url = "http://localhost:11434/api/generate"

payload = {
    "model": "gemma3",
    "prompt": "Say hello",
    "stream": False
}

response = requests.post(url, json=payload)

print(response.status_code)
print(response.json())

200
{'model': 'gemma3', 'created_at': '2026-04-18T13:49:31.0570933Z', 'response': 'Hello there! 😊 How’s it going?', 'done': True, 'done_reason': 'stop', 'context': [105, 2364, 107, 37889, 29104, 106, 107, 105, 4368, 107, 9259, 993, 236888, 103453, 2088, 236858, 236751, 625, 1771, 236881], 'total_duration': 3712860300, 'load_duration': 3490416600, 'prompt_eval_count': 11, 'prompt_eval_duration': 31166000, 'eval_count': 11, 'eval_duration': 121960500}


In [ ]:
from yc_rag_search_gemma import PropertyRAGSearchGemma

# Default model in implementation: gemma3
rag = PropertyRAGSearchGemma(
   # ollama_base_url='http://localhost:11434/api/generate',
   # model_name='gemma3',
    top_k_results=5,
)

print(rag)

langchain_community not found; using SimpleOllamaLLM
Loading gemma3 via Ollama at http://localhost:11434/api/generate ...


RuntimeError: Failed to connect to Ollama at http://localhost:11434/api/generate. Ensure Ollama is running: ollama serve

## 2. Stage 1 Isolated Test - Parameter Extraction

Validate NL-to-JSON extraction quality before evaluating retrieval and generation.

In [ ]:
test_queries = [
    'Find a 4-room flat in Bishan near famous schools under $900k',
    'I want a quiet large flat with good lease left, not too expensive',
    'Show me flats near Nanyang Primary School',
]

for q in test_queries:
    params, fallback, raw = rag._stage1_extract_params(q)
    print(f'Query   : {q}')
    print(f'Params  : {json.dumps(params, indent=2)}')
    print(f'Fallback: {fallback}')
    if fallback and raw:
        print(f'Raw out : {raw[:160]!r}')
    print()

## 3. End-to-End Scenarios

Each scenario runs the full 3-stage RAG pipeline:
- Stage 1: parameter extraction
- Stage 2: Neo4j retrieval
- Stage 3: grounded answer generation

In [ ]:
# --- Scenario 1: Education-focused family ---
result = rag.ask('Find a 4-room flat in Bishan near famous schools under $900k', top_k=5)

print('=' * 64)
print('SCENARIO 1 - Education-focused family')
print('=' * 64)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nFallback used: {result['fallback_used']}")
print()

display_cols = [c for c in [
    'address_key', 'town', 'flat_type', 'floor_area_sqm',
    'resale_price', 'dist_to_nearest_famous_school_km',
    'nearest_famous_school_name', 'composite_score'
] if c in result['results'].columns]

display(result['results'][display_cols])

In [ ]:
# --- Scenario 2: Commuter with budget constraint ---
result = rag.ask('3-room flat near MRT in Toa Payoh under $500k', top_k=5)

print('=' * 64)
print('SCENARIO 2 - Commuter with budget constraint')
print('=' * 64)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"\nAnswer:\n{result['answer']}")
print()

display_cols = [c for c in [
    'address_key', 'town', 'flat_type', 'resale_price',
    'dist_to_mrt_m', 'score_mrt', 'composite_score'
] if c in result['results'].columns]

display(result['results'][display_cols])

In [ ]:
# --- Scenario 3: Graph traversal - near specific famous school ---
result = rag.ask('Show me flats within 1km of Nanyang Primary School', top_k=8)

print('=' * 64)
print('SCENARIO 3 - Graph traversal (near specific school)')
print('=' * 64)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"Special query type: {result['params'].get('special_query_type')}")
print(f"\nAnswer:\n{result['answer']}")
print()

if not result['results'].empty:
    display(result['results'])
else:
    print('(No results - check Neo4j NEAR_FAMOUS_SCHOOL relationships)')

In [ ]:
# --- Scenario 4: Lifestyle criteria - quiet, large, good value ---
result = rag.ask(
    'Large 5-room flat with long lease, affordable price, away from highway noise',
    top_k=5,
)

print('=' * 64)
print('SCENARIO 4 - Lifestyle criteria (quiet, large, value)')
print('=' * 64)
print(f"\nExtracted params:\n{json.dumps(result['params'], indent=2)}")
print(f"\nAnswer:\n{result['answer']}")
print()

display_cols = [c for c in [
    'address_key', 'town', 'flat_type', 'floor_area_sqm',
    'lease_remaining_years', 'resale_price',
    'score_quietness', 'score_size', 'score_lease', 'composite_score'
] if c in result['results'].columns]

display(result['results'][display_cols])

## 4. Error and reliability checks

Production workflows must test ambiguity and no-hit cases to validate fallback quality.

In [ ]:
# --- Edge case 1: Ambiguous query ---
result = rag.ask('???', top_k=5)
print('Query      : ???')
print(f"Fallback   : {result['fallback_used']}")
print(f"Error msg  : {result['error']}")
print(f"Answer     : {result['answer']}")
print()

# --- Edge case 2: Impossible budget ---
result = rag.ask('3-room flat in Bishan under $100k', top_k=5)
print('Query      : 3-room flat in Bishan under $100k')
print(f"Empty      : {result['results'].empty}")
print(f"Answer     : {result['answer']}")

## 5. Interactive search

Edit `user_query` and rerun to test your own scenarios.

In [ ]:
# Edit this query and re-run
user_query = 'Find a 4-room flat near Tampines with famous school and MRT access'

result = rag.ask(user_query, top_k=5)

print(f'Query  : {user_query}')
print('\nExtracted params:')
print(json.dumps(result['params'], indent=2))
print(f"\nAnswer:\n{result['answer']}")
print(f"\nFallback used: {result['fallback_used']}")
print()
display(result['results'])

In [ ]:
# --- Cleanup ---
rag.close()
print('Neo4j connection closed.')